In [1]:
# 5. Evaluation of the LLM annotation

import json
from collections import Counter
from pathlib import Path

import pandas as pd

DATA = Path("../data")   # not included in the repository
TAB = Path("../tab")

with open(DATA / "llm" / "geluid_annotaties_predictions.json") as f:
    tasks = json.load(f)

LABELS = ["doorbell ringing", "singing", "making noise", "disrupting nightrest", "playing inn music",
          "scolding", "yelling", "arguing", "playing street music", "traffic noise"]   # 'other sound' excluded
print(len(tasks), "incidents annotated")

15683 incidents annotated


In [2]:
def spans(blocks, exclude=()):
    """Labelled spans of the first prediction or annotation block of a task."""
    return [r for b in blocks[:1] for r in b["result"] if set(r["value"]["labels"]) - set(exclude)]

def labels_in(blocks):
    return {label for r in spans(blocks) for label in r["value"]["labels"]}

for name, key in [("predicted", "predictions"), ("corrected", "annotations")]:
    n = [len(spans(t[key])) for t in tasks]
    print(f"{name:<10} {sum(n):,} mentions in {sum(x > 0 for x in n):,} incidents")

assert not any(spans(t["predictions"]) and not t["annotations"] for t in tasks), "unreviewed predictions"

predicted  3,730 mentions in 2,804 incidents
corrected  2,756 mentions in 1,712 incidents


In [3]:
# Table 3

pairs = [(labels_in(t["annotations"]), labels_in(t["predictions"])) for t in tasks if t["annotations"]]

def prf(tp, fp, fn):
    p, r = tp / (tp + fp), tp / (tp + fn)
    return p, r, 2 * p * r / (p + r)

rows, total = [], Counter()
for label in LABELS:
    tp = sum(label in gold and label in pred for gold, pred in pairs)
    fp = sum(label in pred and label not in gold for gold, pred in pairs)
    fn = sum(label in gold and label not in pred for gold, pred in pairs)
    total.update(tp=tp, fp=fp, fn=fn)
    rows.append([label, tp + fp, tp + fn, *prf(tp, fp, fn)])

table3 = pd.DataFrame(rows, columns=["label", "predictions", "corrections", "precision", "recall", "f1"])
table3 = table3.sort_values("f1", ascending=False).reset_index(drop=True)
table3.loc[len(table3)] = ["macro mean", table3["predictions"].sum(), table3["corrections"].sum(),
                           *table3[["precision", "recall", "f1"]].mean()]
table3.loc[len(table3)] = ["micro mean", total["tp"] + total["fp"], total["tp"] + total["fn"],
                           *prf(total["tp"], total["fp"], total["fn"])]
table3.round(3).to_csv(TAB / "table3_llm_evaluation.csv", index=False)
table3.round(3)

,label,predictions,corrections,precision,recall,f1
0,doorbell ringing,213,180,0.803,0.950,0.870
1,singing,152,205,0.961,0.712,0.818
2,making noise,530,422,0.711,0.893,0.792
3,disrupting nightrest,459,447,0.765,0.785,0.775
4,playing inn music,66,79,0.833,0.696,0.759
5,scolding,108,189,0.926,0.529,0.673
6,yelling,921,514,0.496,0.889,0.637
7,arguing,111,226,0.901,0.442,0.593
8,playing street music,27,21,0.519,0.667,0.583
9,traffic noise,29,34,0.414,0.353,0.381
